
# 応用問題 (集大成): 多層パーセプトロン (MLP) を自分で学習させる

## 目標

AI の「学習」の中身が実は **行列積の繰り返し** であることを, **多層パーセプトロン (MLP)** を一から実装して体験する。**forward (順伝播) → 損失 → backprop (逆伝播) → SGD (更新)** のループを自分で書き, **非線形な分類境界**を学習させる。

前の問題 (ロジスティック回帰) は線形分離可能なデータだった。今回は **線形では分けられない** データ (円の内外) を扱うので, **隠れ層** が必須である。

並列化対象は AI 学習の心臓部, **全サンプルにわたる勾配の和** である。

## 課題

合成データ (非線形分離):

- 各サンプルの座標を `[-1,1]^2` から乱数で生成 (`draw_rand01`)。
- ラベル `y_i = (x0^2 + x1^2 < R^2) ? 1 : 0` (**内側の円が class 1**)。直線では分けられない。

ネットワーク (入力 2 → 隠れ層 H → 出力 1):

- forward: `h_k = tanh(Σ_d W1[k,d] x_d + b1[k])`,  `o = sigmoid(Σ_k W2[k] h_k + b2)`
- 損失: 二値クロスエントロピー
- backprop: `do = o - y`,  `dW2[k] += do·h_k`,  `dh_k = do·W2[k]·(1 - h_k^2)`,  `dW1[k,d] += dh_k·x_d`, ...
- 更新 (フルバッチ勾配降下): 全サンプルにわたって勾配を**総和**し, `W -= lr·(勾配/N)`。

各エポックで一番重いのは **全 `N` サンプルにわたる forward + backprop の O(N·H)**。各サンプルの寄与は独立なので並列化できる。ただし勾配配列 `gW1, gb1, gW2, gb2` への加算は競合するので, **配列 reduction** で安全に総和する:

- C++: `#pragma omp parallel for reduction(+:loss,correct,gb2,gW1[:H*D],gb1[:H],gW2[:H])`
- Fortran: `!$omp parallel do private(...) reduction(+:loss,correct,gb2,gW1,gb1,gW2)` … `!$omp end parallel do` (サンプルごとの一時配列 `hh` は `private`)

これが `TODO` の並列化箇所である。スレッド数を変えても結果 (正解率) は同じになることを確認せよ。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mlp_train.cpp -o mlp_train.exe

# Fortran
nvfortran -fast -mp=multicore mlp_train.f90 -o mlp_train.exe
```

引数: サンプル数 `N` (既定 4000), 隠れ層ユニット数 `H` (既定 32), エポック数 `E` (既定 3000), 学習率 `lr` (既定 0.7)。

```
OMP_NUM_THREADS=4 ./mlp_train.exe 4000 32 3000 0.7
```

## 期待される結果

```
epoch    0: loss=0.6951, acc=49.67%
epoch  500: loss=0.1388, acc=96.67%
...
epoch 2999: loss=0.0420, acc=99.38%
最終: N=4000, H=32, epochs=3000, loss=0.0420, acc=99.38%
elapsed = ... sec
```

- 学習が進むと **損失が下がり, 正解率が上がる**。隠れ層のおかげで非線形な円の境界を学習し, **最終的に 95% を大きく超える** (ほぼ 99%)。
- `OMP_NUM_THREADS` を増やすと `elapsed` が短くなる (台数効果)。正解率は本質的に同じになる。
- (発展) 内側の行列積を SIMD 化, あるいは GPU にオフロードして更に高速化できる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mlp_train.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (合成データ・初期値生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

static inline double sigmoidf(double z) { return 1.0 / (1.0 + exp(-z)); }

/* 多層パーセプトロン (MLP) を自分で学習させる。
   ネットワーク: 入力 2 -> 隠れ層 H (tanh) -> 出力 1 (sigmoid)。
   2次元データの二値分類で, クラス境界が「円」(非線形分離) なので
   隠れ層が必須。AI の「学習」の中身が行列積であることを体験する。

   forward:  h_k = tanh( Σ_d W1[k,d] x_d + b1[k] ),  o = sigmoid( Σ_k W2[k] h_k + b2 )
   損失:     二値クロスエントロピー
   backprop: do = o - y,  dW2[k] += do * h_k,  db2 += do,
             dh_k = do * W2[k] * (1 - h_k^2),
             dW1[k,d] += dh_k * x_d,  db1[k] += dh_k
   更新:     全サンプルにわたる勾配の和を取り, W -= lr * grad/N。
   並列化対象は「全サンプルにわたる勾配の和」(配列 reduction)。 */
int main(int argc, char ** argv) {
  long N = (argc > 1 ? atol(argv[1]) : 4000);   /* サンプル数 */
  int  H = (argc > 2 ? atoi(argv[2]) : 32);     /* 隠れ層のユニット数 */
  int  E = (argc > 3 ? atoi(argv[3]) : 3000);   /* エポック数 */
  double lr = (argc > 4 ? atof(argv[4]) : 0.7); /* 学習率 */
  const int D = 2;                              /* 入力次元 */
  const double R2 = 0.5;                        /* 内側の円の半径^2 */

  /* 合成データ: [-1,1]^2 上の点。原点に近い (内側の円) なら class 1。非線形分離。 */
  double * X = (double *)malloc(sizeof(double) * N * D);
  int    * y = (int *)malloc(sizeof(int) * N);
  for (long i = 0; i < N; i++) {
    double x0 = draw_rand01(i, 0) * 2.0 - 1.0;
    double x1 = draw_rand01(i, 1) * 2.0 - 1.0;
    X[i*D+0] = x0; X[i*D+1] = x1;
    y[i] = (x0*x0 + x1*x1 < R2) ? 1 : 0;
  }

  /* パラメータ: W1[H][D], b1[H], W2[H], b2。小さな乱数で初期化。 */
  double * W1 = (double *)malloc(sizeof(double) * H * D);
  double * b1 = (double *)malloc(sizeof(double) * H);
  double * W2 = (double *)malloc(sizeof(double) * H);
  double   b2 = 0.0;
  for (int k = 0; k < H; k++) {
    for (int d = 0; d < D; d++) W1[k*D+d] = (draw_rand01(k, d+10) - 0.5);
    b1[k] = 0.0;
    W2[k] = (draw_rand01(k, 99) - 0.5);
  }

  /* 勾配の総和を入れる配列 */
  double * gW1 = (double *)malloc(sizeof(double) * H * D);
  double * gb1 = (double *)malloc(sizeof(double) * H);
  double * gW2 = (double *)malloc(sizeof(double) * H);

  double loss = 0.0; long correct = 0;
  double t0 = omp_get_wtime();
  for (int ep = 0; ep < E; ep++) {
    for (int k = 0; k < H; k++) { gW1[k*D+0]=0; gW1[k*D+1]=0; gb1[k]=0; gW2[k]=0; }
    double gb2 = 0.0;
    loss = 0.0; correct = 0;

    /* 全サンプルにわたる forward + backprop。各サンプルの勾配寄与を総和する。
       損失・正解数はスカラ reduction, 勾配は配列 reduction で競合を避ける。 */
    // TODO: サンプルのループを配列 reduction で並列化せよ: #pragma omp parallel for reduction(+:loss,correct,gb2,gW1[:H*D],gb1[:H],gW2[:H]).
    for (long i = 0; i < N; i++) {
      double x0 = X[i*D+0], x1 = X[i*D+1];
      /* forward */
      double o_in = b2;
      double h[256];                     /* H <= 256 を仮定 */
      for (int k = 0; k < H; k++) {
        double z = b1[k] + W1[k*D+0]*x0 + W1[k*D+1]*x1;
        double hk = tanh(z);
        h[k] = hk;
        o_in += W2[k] * hk;
      }
      double o = sigmoidf(o_in);
      double yi = (double)y[i];
      double eps = 1e-12;
      loss -= (y[i] ? log(o + eps) : log(1.0 - o + eps));
      if (((o > 0.5) ? 1 : 0) == y[i]) correct++;
      /* backprop */
      double dout = o - yi;
      gb2 += dout;
      for (int k = 0; k < H; k++) {
        gW2[k] += dout * h[k];
        double dh = dout * W2[k] * (1.0 - h[k]*h[k]);
        gW1[k*D+0] += dh * x0;
        gW1[k*D+1] += dh * x1;
        gb1[k]     += dh;
      }
    }
    loss /= (double)N;

    /* 更新 (勾配を平均して降下) */
    double s = lr / (double)N;
    for (int k = 0; k < H; k++) {
      W1[k*D+0] -= s * gW1[k*D+0];
      W1[k*D+1] -= s * gW1[k*D+1];
      b1[k]     -= s * gb1[k];
      W2[k]     -= s * gW2[k];
    }
    b2 -= s * gb2;

    if (ep % 500 == 0 || ep == E - 1)
      printf("epoch %4d: loss=%.4f, acc=%.2f%%\n", ep, loss, 100.0 * correct / N);
  }
  double elapsed = omp_get_wtime() - t0;

  printf("最終: N=%ld, H=%d, epochs=%d, loss=%.4f, acc=%.2f%%\n",
         N, H, E, loss, 100.0 * correct / N);
  printf("elapsed = %.3f sec\n", elapsed);
  free(X); free(y); free(W1); free(b1); free(W2); free(gW1); free(gb1); free(gW2);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mlp_train.cpp -o mlp_train_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mlp_train_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mlp_train.f90
module mlp_mod
contains
  ! 状態を持たない乱数 (合成データ・初期値生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  function sigmoidf(z) result(s)
    real(8), intent(in) :: z
    real(8) :: s
    s = 1.0d0 / (1.0d0 + exp(-z))
  end function sigmoidf
end module mlp_mod

! 多層パーセプトロン (MLP) を自分で学習させる。
! ネットワーク: 入力 2 -> 隠れ層 H (tanh) -> 出力 1 (sigmoid)。
! 2次元データの二値分類。境界が「円」(非線形分離) なので隠れ層が必須。
! forward -> 損失 -> backprop -> 更新 を繰り返す。
! 並列化対象は「全サンプルにわたる勾配の和」(配列 reduction)。
program mlp_train
  use mlp_mod
  use omp_lib
  character(len=32) :: arg
  integer, parameter :: D = 2
  integer :: H, E, ep, k, d2
  integer(8) :: N8, i
  real(8) :: lr, R2, loss, x0, x1, o_in, o, yi, dout, dh, z, hk, s, gb2, b2, elapsed, t0
  integer(8) :: correct
  real(8), allocatable :: X(:), W1(:), b1(:), W2(:), gW1(:), gb1(:), gW2(:), hh(:)
  integer, allocatable :: y(:)

  N8 = 4000; H = 32; E = 3000; lr = 0.7d0; R2 = 0.5d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N8
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) H
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) E
  end if
  if (command_argument_count() >= 4) then
     call get_command_argument(4, arg); read (arg, *) lr
  end if

  ! 合成データ: [-1,1]^2 上の点。原点に近ければ class 1。非線形分離。
  allocate(X(0:N8*D-1), y(0:N8-1))
  do i = 0, N8 - 1
     x0 = draw_rand01(i, 0_8) * 2.0d0 - 1.0d0
     x1 = draw_rand01(i, 1_8) * 2.0d0 - 1.0d0
     X(i*D+0) = x0; X(i*D+1) = x1
     if (x0*x0 + x1*x1 < R2) then
        y(i) = 1
     else
        y(i) = 0
     end if
  end do

  ! パラメータ: W1[H,D], b1[H], W2[H], b2。小さな乱数で初期化。
  allocate(W1(0:H*D-1), b1(0:H-1), W2(0:H-1), gW1(0:H*D-1), gb1(0:H-1), gW2(0:H-1))
  do k = 0, H - 1
     do d2 = 0, D - 1
        W1(k*D+d2) = draw_rand01(int(k,8), int(d2+10,8)) - 0.5d0
     end do
     b1(k) = 0.0d0
     W2(k) = draw_rand01(int(k,8), 99_8) - 0.5d0
  end do
  b2 = 0.0d0

  loss = 0.0d0; correct = 0
  t0 = omp_get_wtime()
  do ep = 0, E - 1
     gW1 = 0.0d0; gb1 = 0.0d0; gW2 = 0.0d0; gb2 = 0.0d0
     loss = 0.0d0; correct = 0

     ! 全サンプルにわたる forward + backprop。各サンプルの勾配寄与を総和する。
     ! 損失・正解数はスカラ reduction, 勾配は配列 reduction で競合を避ける。
     ! TODO: サンプルのループを配列 reduction で並列化せよ: !$omp parallel do private(...) reduction(+:loss,correct,gb2,gW1,gb1,gW2) (h は private)。
     do i = 0, N8 - 1
        allocate(hh(0:H-1))
        x0 = X(i*D+0); x1 = X(i*D+1)
        ! forward
        o_in = b2
        do k = 0, H - 1
           z = b1(k) + W1(k*D+0)*x0 + W1(k*D+1)*x1
           hk = tanh(z)
           hh(k) = hk
           o_in = o_in + W2(k) * hk
        end do
        o = sigmoidf(o_in)
        yi = real(y(i), 8)
        if (y(i) == 1) then
           loss = loss - log(o + 1.0d-12)
        else
           loss = loss - log(1.0d0 - o + 1.0d-12)
        end if
        if (merge(1, 0, o > 0.5d0) == y(i)) correct = correct + 1
        ! backprop
        dout = o - yi
        gb2 = gb2 + dout
        do k = 0, H - 1
           gW2(k) = gW2(k) + dout * hh(k)
           dh = dout * W2(k) * (1.0d0 - hh(k)*hh(k))
           gW1(k*D+0) = gW1(k*D+0) + dh * x0
           gW1(k*D+1) = gW1(k*D+1) + dh * x1
           gb1(k)     = gb1(k) + dh
        end do
        deallocate(hh)
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     loss = loss / real(N8, 8)

     ! 更新 (勾配を平均して降下)
     s = lr / real(N8, 8)
     do k = 0, H - 1
        W1(k*D+0) = W1(k*D+0) - s * gW1(k*D+0)
        W1(k*D+1) = W1(k*D+1) - s * gW1(k*D+1)
        b1(k)     = b1(k) - s * gb1(k)
        W2(k)     = W2(k) - s * gW2(k)
     end do
     b2 = b2 - s * gb2

     if (mod(ep, 500) == 0 .or. ep == E - 1) then
        print "(a,i4,a,f7.4,a,f6.2,a)", "epoch ", ep, ": loss=", loss, &
             ", acc=", 100.0d0 * correct / N8, "%"
     end if
  end do
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,i0,a,f7.4,a,f6.2,a)", "最終: N=", N8, ", H=", H, &
       ", epochs=", E, ", loss=", loss, ", acc=", 100.0d0 * correct / N8, "%"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program mlp_train

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mlp_train.f90 -o mlp_train_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mlp_train_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mlp_train.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mlp_train.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mlp_train.cpp}